# 05 — Classification: hyperparameter tuning

Tunes eight classifiers with Optuna, each maximising **Macro F1** over 5-fold
stratified cross-validation.

Macro F1 rather than accuracy, because ~69% of children are class 0: a model
that always predicts "None" would score 69% accuracy and be useless. Macro F1
weights all four classes equally, so ignoring the rare ones is punished.

Every model also handles imbalance directly, through `class_weight='balanced'`
or per-sample weights inside the folds.

As in 03, the scaler, imputer and PCA are re-fitted inside each fold, and each
study writes to `optuna trials/<model>_clf_trials.csv` for notebook 06 to read.

Equivalent to: `python main.py --stage tune-clf`

In [1]:
# Make the project's src/ package importable from inside notebooks/.
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

import warnings

from src.config import IMPUTER_PARAMS_CLF, PATHS, STUDY_TRIALS
from src.data_loader import DROP_FOR_CLASSIFICATION, load_processed_split
from src.tuning import tune_model

warnings.filterwarnings("ignore")
os.makedirs(PATHS["results_dir"], exist_ok=True)

# Load the dataset built by 02b. Features stay RAW — imputation and scaling
# happen inside each CV fold.
X_train_raw, y_train_clf, _, _ = load_processed_split(
    PATHS["clf_processed"], DROP_FOR_CLASSIFICATION
)

print(f"X_train_raw shape: {X_train_raw.shape} "
      f"(NaNs present: {X_train_raw.isnull().any().any()})")
print(f"Active imputer configuration: {IMPUTER_PARAMS_CLF}")

/Users/alessandro/miniforge3/envs/m4_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


X_train_raw shape: (6768, 46) (NaNs present: True)
Active imputer configuration: {'imputer_choice': 'mice', 'mice_max_iter': 6, 'mice_initial_strategy': 'median', 'mice_et_estimators': 19, 'mice_et_max_depth': 5}


## LightGBM — 50 trials

Gradient-boosted trees with `class_weight='balanced'`.

In [2]:
study_lgbm = tune_model("lgbm", "clf", X_train_raw, y_train_clf,
                        IMPUTER_PARAMS_CLF, PATHS["results_dir"])

Starting LightGBM optimization (maximizing Macro F1)...


Best trial: 42. Best value: 0.348665: 100%|██████████| 50/50 [1:38:33<00:00, 118.27s/it]

LightGBM: Best trial CV Macro F1: 0.3487


## XGBoost — 50 trials

Boosting with a softmax objective over the four classes.

In [3]:
study_xgb = tune_model("xgb", "clf", X_train_raw, y_train_clf,
                        IMPUTER_PARAMS_CLF, PATHS["results_dir"])

Starting XGBoost optimization (maximizing Macro F1)...


Best trial: 28. Best value: 0.364267: 100%|██████████| 50/50 [1:12:57<00:00, 87.55s/it]

XGBoost: Best trial CV Macro F1: 0.3643


## CatBoost — 50 trials

Boosting with `auto_class_weights='Balanced'`.

In [4]:
study_cb = tune_model("cb", "clf", X_train_raw, y_train_clf,
                        IMPUTER_PARAMS_CLF, PATHS["results_dir"])

Starting CatBoost optimization (maximizing Macro F1)...


Best trial: 45. Best value: 0.368315: 100%|██████████| 50/50 [1:47:10<00:00, 128.60s/it]

CatBoost: Best trial CV Macro F1: 0.3683


## Random Forest — 50 trials

Bagged trees; the strongest model by cross-validated Macro F1.

In [5]:
study_rf = tune_model("rf", "clf", X_train_raw, y_train_clf,
                        IMPUTER_PARAMS_CLF, PATHS["results_dir"])

Starting RandomForest optimization (maximizing Macro F1)...


Best trial: 47. Best value: 0.287214: 100%|██████████| 50/50 [1:12:10<00:00, 86.61s/it]

RandomForest: Best trial CV Macro F1: 0.2872


## Ridge Classifier — 100 trials

Linear baseline with PCA. Given a larger budget because its search space is small and cheap to explore.

In [6]:
study_ridge = tune_model("ridge", "clf", X_train_raw, y_train_clf,
                        IMPUTER_PARAMS_CLF, PATHS["results_dir"])

Starting Ridge optimization (maximizing Macro F1)...


Best trial: 35. Best value: 0.0342359: 100%|██████████| 100/100 [2:24:17<00:00, 86.58s/it] 

Ridge: Best trial CV Macro F1: 0.0342


## SVC — 50 trials

Kernel classifier with PCA; `probability=True` so it can feed the stacking ensemble.

In [ ]:
study_svc = tune_model("svc", "clf", X_train_raw, y_train_clf,
                        IMPUTER_PARAMS_CLF, PATHS["results_dir"])

Starting SVC optimization (maximizing Macro F1)...


Best trial: 31. Best value: 0.285003:  68%|██████▊   | 34/50 [1:18:02<34:07, 127.99s/it]

## Logistic Regression — 100 trials

Linear baseline with PCA. The solver is itself a search parameter.

In [2]:
study_lr = tune_model("lr", "clf", X_train_raw, y_train_clf,
                        IMPUTER_PARAMS_CLF, PATHS["results_dir"])

Starting LogisticRegression optimization (maximizing Macro F1)...


Best trial: 61. Best value: 0.313126:  76%|███████▌  | 76/100 [1:18:53<24:54, 62.29s/it]


[W 2026-08-03 20:21:00,326] Trial 76 failed with parameters: {'C': 0.44932109558855604, 'penalty': 'l2', 'solver': 'newton-cg', 'pca_n_components': 17} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/alessandro/miniforge3/envs/m4_env/lib/python3.13/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/Users/alessandro/Documents/NEW PROJECT/Local_DM2_Project/tabular/src/tuning.py", line 504, in objective
    return cross_validated_score(params, model_class, X_train_raw, y_train,
                                 imputer_params, task, n_components)
  File "/Users/alessandro/Documents/NEW PROJECT/Local_DM2_Project/tabular/src/tuning.py", line 170, in cross_validated_score
    fold_scores = Parallel(n_jobs=get_n_jobs())(
        delayed(evaluate_fold)(train_idx, valid_idx, X_raw, y, params,
                               model_class, imputer_params, task, n_components)
        for t

KeyboardInterrupt: 

## PyTorch MLP — 100 trials

The neural network — and the model explained in notebook 06.

In [3]:
study_nn = tune_model("nn", "clf", X_train_raw, y_train_clf,
                        IMPUTER_PARAMS_CLF, PATHS["results_dir"])

Starting TorchMLP optimization (maximizing Macro F1)...


Best trial: 54. Best value: 0.338268: 100%|██████████| 100/100 [4:56:06<00:00, 177.67s/it] 

TorchMLP: Best trial CV Macro F1: 0.3383
